In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json, pickle, pathlib, warnings
warnings.filterwarnings("ignore")

In [2]:
ROOT = pathlib.Path("..").resolve()
DATA = ROOT / "data" / "day03_ready" / "tar" / "02084000"
PRED_DIR = ROOT / "data" / "predictions"
FIG_DIR  = ROOT / "figures" / "day05"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [15]:
with open(ROOT / "data" / "day03_ready" / "model_config.json") as f:
    cfg = json.load(f)
tar_cfg = cfg["targets"]["tar_02084000"]

with open(ROOT / "data" / "day03_ready" / "norm_params.pkl", "rb") as f:
    nparams = pickle.load(f)["tar_02084000"]

In [22]:
HORIZONS = tar_cfg['horizons']
FLOOD_THRESHOLD = cfg['flood_thresholds']['02084000']
FLOOD_WEIGHT = 10.0 

print('Days for Forecasting: ', f'{HORIZONS}')
print('Flood Threshold in Cubic Flow per Second: ', f'{FLOOD_THRESHOLD}')

Days for Forecasting:  [1, 2, 3, 4]
Flood Threshold in Cubic Flow per Second:  8641.499999999996


In [18]:
mean_lf = nparams["mean"]["log_flow"]
std_lf  = nparams["std"]["log_flow"]

In [26]:
# local columns
local_cols = ["log_flow_norm", "prcp_mm_norm", "prcp_7day_norm", "prcp_14day_norm"]

# load splits 
splits = {}
for name in ['train', 'val', 'test']:
    splits[name] = pd.read_parquet(DATA / f"{name}.parquet")
    
# computing the normalized targets 
for name, df in splits.items():
    for h in HORIZONS: 
        df[f"target_norm_t{h}"] = ((df[f"target_log_flow_t{h}"] - mean_lf) / std_lf).astype("float32")

# upstream colums 
up_flow_cols = sorted([c for c in splits["train"].columns if c.startswith("up_flow_") and c.endswith("_norm")])
basin_cols   = sorted([c for c in splits["train"].columns if c.startswith("basin_prcp_") and c.endswith("_norm")])
upstream_cols = local_cols + up_flow_cols + basin_cols

print(f"Local features:    {len(local_cols)}   → A_local  will be n × {len(local_cols)+1} (with bias)")
print(f"Upstream features: {len(upstream_cols)} → A_upstream will be n × {len(upstream_cols)+1} (with bias)")

Local features:    4   → A_local  will be n × 5 (with bias)
Upstream features: 25 → A_upstream will be n × 26 (with bias)


In [29]:
def build_matrix(df, feature_cols, target_cols):
    mask = df[target_col].notna()
    valid = df[mask]

    X = valid[feature_cols].values.astype(np.float64)
    b = valid[target_cols].values.astype(np.float64)

    X = np.nan_to_num(X, nan = 0.0)

    # Add bias column
    n = X.shape[0]
    A = np.column_stack([np.ones(n), X])

    return A, b, valid.index

matrices = {} 
model_features = {
    "local": local_cols,
    "upstream": upstream_cols,
}

for model_type, feat_cols in model_features.items():
    for h in HORIZONS:
        target_col = f"target_norm_t{h}"
        A, b, idx = build_matrix(splits["train"], feat_cols, target_col)
        matrices[(model_type, h)] = {"A": A, "b": b, "idx": idx}

print(f"  {'Model':<12} {'Horizon':<10} {'A shape':<18} {'b shape':<12} {'Rank check'}")
print(f"  {'─'*12} {'─'*10} {'─'*18} {'─'*12} {'─'*12}")
for (mt, h), m in sorted(matrices.items()):
    A = m["A"]
    rank = np.linalg.matrix_rank(A)
    print(f"  {mt:<12} t+{h:<8} {str(A.shape):<18} {str(m['b'].shape):<12} rank={rank}/{A.shape[1]}")

  Model        Horizon    A shape            b shape      Rank check
  ──────────── ────────── ────────────────── ──────────── ────────────
  local        t+1        (7178, 5)          (7178,)      rank=5/5
  local        t+2        (7178, 5)          (7178,)      rank=5/5
  local        t+3        (7178, 5)          (7178,)      rank=5/5
  local        t+4        (7178, 5)          (7178,)      rank=5/5
  upstream     t+1        (7178, 26)         (7178,)      rank=26/26
  upstream     t+2        (7178, 26)         (7178,)      rank=26/26
  upstream     t+3        (7178, 26)         (7178,)      rank=26/26
  upstream     t+4        (7178, 26)         (7178,)      rank=26/26


In [48]:
# Solving with 2 equations 
solutions_ols = {}
for (model_type, h), m in sorted(matrices.items()):
    A, b = m["A"], m["b"]

    # Normal equations  →  (A^T A) x = A^T b
    x_normal = np.linalg.solve((A.T @ A), (A.T @ b))
    
    # QR Decomp of A 
    Q, R = np.linalg.qr(A)
    x_qr = np.linalg.solve(R, Q.T @ b)

    agree_nq = np.allclose(x_normal, x_qr, atol=1e-10)
    solutions_ols[(model_type, h)] = x_qr

    print(f"{model_type:>10} t+{h}: Normal≈QR: {agree_nq} "
          f"‖residual‖² = {np.sum((A @ x_qr - b)**2):.4f}")

     local t+1: Normal≈QR: True ‖residual‖² = 639.9413
     local t+2: Normal≈QR: True ‖residual‖² = 872.5679
     local t+3: Normal≈QR: True ‖residual‖² = 1279.1320
     local t+4: Normal≈QR: True ‖residual‖² = 1679.0077
  upstream t+1: Normal≈QR: True ‖residual‖² = 417.0216
  upstream t+2: Normal≈QR: True ‖residual‖² = 547.8246
  upstream t+3: Normal≈QR: True ‖residual‖² = 819.5849
  upstream t+4: Normal≈QR: True ‖residual‖² = 1160.1140


In [51]:
# weighted least squares
# Standard OLS weights everyday equally 
# Weighted LS: multiply each row by sqrt(w_i), then solve the transformed system
solutions_wls = {}   # (model_type, horizon) → x_wls

for (model_type, h), m in sorted(matrices.items()):
    A, b, idx = m["A"], m["b"], m["idx"]
    log_flow_target = b * std_lf + mean_lf # unnormalize
    flow_cfs = np.expm1(log_flow_target)
    is_flood = (flow_cfs > FLOOD_THRESHOLD).astype(float)

    w = np.where(is_flood > 0.5, np.sqrt(FLOOD_WEIGHT), 1.0)
    
    #   A_w = W^{1/2} A,  b_w = W^{1/2} b
    A_w = A * w[:, np.newaxis]
    b_w = b * w

    x_wls, _, _, _ = np.linalg.lstsq(A_w, b_w, rcond=None)
    solutions_wls[(model_type, h)] = x_wls

    n_flood = int(is_flood.sum())
    x_ols = solutions_ols[(model_type, h)]
    max_diff = np.max(np.abs(x_ols - x_wls))

    print(f"{model_type:>10} t+{h}:  flood_days={n_flood:>4}/{len(b)}  "
          f"max|x_ols - x_wls| = {max_diff:.4f}")
    

     local t+1:  flood_days= 359/7178  max|x_ols - x_wls| = 0.0273
     local t+2:  flood_days= 359/7178  max|x_ols - x_wls| = 0.0408
     local t+3:  flood_days= 359/7178  max|x_ols - x_wls| = 0.0640
     local t+4:  flood_days= 359/7178  max|x_ols - x_wls| = 0.1040
  upstream t+1:  flood_days= 359/7178  max|x_ols - x_wls| = 0.0470
  upstream t+2:  flood_days= 359/7178  max|x_ols - x_wls| = 0.1080
  upstream t+3:  flood_days= 359/7178  max|x_ols - x_wls| = 0.1331
  upstream t+4:  flood_days= 359/7178  max|x_ols - x_wls| = 0.0899


In [ ]:
# ŷ_norm = A @ x_wls
def predict(df, x_wls, feat_cols):
    target_col = f"target_norm_t{horizon}"
    target_raw_col = f"target_log_flow_t{horizon}"

    mask = df[target_col].notna()
    valid = df[mask]

    X = valid[feat_cols].values.astype(np.float64)
    X = np.nan_to_num(X, nan=0.0)
    A = np.column_stack([np.ones(len(X)), X])

    y_pred_norm = A @ x_wls
    log_predicted = y_pred_norm * std_lf + mean_lf
    predicted_cfs = np.expm1(log_predicted)
    predicted_cfs = np.maximum(predicted_cfs, 0.0) # non negatives

    log_observed = valid[target_raw_col].values
    observed_cfs = np.expm1(log_observed)

    is_flood = (observed_cfs > FLOOD_THRESHOLD).astype(int)
    
    # Network state: count upstream gauges above their flood thresholds
    up_gauges = tar_cfg["upstream_gauges"]
    net_state = np.zeros(len(valid), dtype=int)
    for ug in up_gauges:
        col = f"up_flow_{ug}"
        if col in valid.columns:
            ug_thresh = cfg["flood_thresholds"].get(ug, 1e9)
            ug_flow = np.expm1(valid[col].values)
            net_state += (ug_flow > ug_thresh).astype(int)
    
    # Build output dataframe
    out = pd.DataFrame({
        "date": valid.index,
        "observed_cfs": observed_cfs,
        "predicted_cfs": predicted_cfs,
        "log_observed": log_observed,
        "log_predicted": log_predicted,
        "is_flood": is_flood,
        "network_state": net_state,
        "horizon": horizon,
    })

predictions = {}   # (model_type, horizon, split) → DataFrame

for model_type, feat_cols in model_features.items():
    for h in HORIZONS:
        x_wls = solutions_wls[(model_type, h)]
        for split_name in ["val", "test"]:
            df = splits[split_name]
            out = predict_and_save(split_name, df, model_type, h, x_wls, feat_cols)
            predictions[(model_type, h, split_name)] = out
    